# Agrupamento de usuários e classificação de perfis do SoulUp

Este notebook treina três modelos com a base `dataset/soulup_dataset.csv`: K-Means para agrupar usuários com padrões de missões semelhantes e Random Forest para classificar o perfil de referência. Os grupos do K-Means são formados sem usar os rótulos; os perfis de referência são usados depois apenas para interpretar e avaliar os grupos.

A base contém 400 registros e cinco perfis de referência. Os resultados descrevem somente os dados deste CSV, cuja origem não está documentada, e não garantem o mesmo desempenho com usuários reais ou dados futuros.

In [2]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    adjusted_rand_score,
    classification_report,
    confusion_matrix,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DIRETORIO = Path.cwd()
if not (DIRETORIO / "dataset" / "soulup_dataset.csv").is_file():
    DIRETORIO = DIRETORIO / "ia"

ARQUIVO_DADOS = DIRETORIO / "dataset" / "soulup_dataset.csv"
ARQUIVO_MODELOS = DIRETORIO / "modelos_soulup.joblib"
if not ARQUIVO_DADOS.is_file():
    raise FileNotFoundError("Não foi encontrado dataset/soulup_dataset.csv.")

FEATURES_CATEGORIAS = [
    "missoes_reciclagem",
    "missoes_mobilidade",
    "missoes_agua",
    "missoes_energia",
    "missoes_natureza",
    "missoes_consumo",
]
FEATURES_CLUSTER = FEATURES_CATEGORIAS + ["taxa_conclusao"]
FEATURES_PERFIL = FEATURES_CATEGORIAS
COLUNA_ALVO = "perfil_referencia"

df = pd.read_csv(ARQUIVO_DADOS)
colunas_necessarias = FEATURES_CLUSTER + [COLUNA_ALVO]
colunas_ausentes = []
for coluna in colunas_necessarias:
    if coluna not in df.columns:
        colunas_ausentes.append(coluna)
if colunas_ausentes:
    raise ValueError("Colunas ausentes na base: " + ", ".join(colunas_ausentes))
if df[colunas_necessarias].isnull().any().any():
    raise ValueError("A base possui valores ausentes nas colunas usadas pelos modelos.")

print("Registros:", len(df))
print("Distribuição dos perfis de referência:")
print(df[COLUNA_ALVO].value_counts().sort_index().to_string())
print("\nPrimeiros registros usados:")
print(df[FEATURES_CLUSTER + [COLUNA_ALVO]].head().to_string(index=False))

Registros: 400
Distribuição dos perfis de referência:
perfil_referencia
CONSUMO_CONSCIENTE        80
EXPLORADOR_SUSTENTAVEL    80
GUARDIAO_RECICLAGEM       80
GUARDIAO_RECURSOS         80
MOBILIZADOR_VERDE         80

Primeiros registros usados:
 missoes_reciclagem  missoes_mobilidade  missoes_agua  missoes_energia  missoes_natureza  missoes_consumo  taxa_conclusao      perfil_referencia
                  9                   2             2                3                 3                3           0.864    GUARDIAO_RECICLAGEM
                  2                  10             2                3                 2                2           0.762      MOBILIZADOR_VERDE
                  2                   2            11                9                 2                3           0.828      GUARDIAO_RECURSOS
                  3                   2             1                2                 1                7           0.750     CONSUMO_CONSCIENTE
                  6          

## Modelo 1 — K-Means para agrupar usuários

K-Means foi escolhido para separar os usuários em grupos de acordo com seus padrões de missões. São usadas as seis quantidades de missões por categoria e a taxa de conclusão. Como esses campos têm escalas diferentes, `StandardScaler` é ajustado antes de calcular os grupos. Foram definidos cinco grupos por serem cinco os perfis de referência existentes na base; o algoritmo, porém, não recebe esses rótulos ao treinar.

A silhueta avalia a separação entre os grupos. O Adjusted Rand Index (ARI) compara os grupos encontrados aos rótulos de referência sem depender dos números arbitrários atribuídos a cada grupo. Para facilitar a interpretação, cada grupo recebe o nome do perfil de referência mais frequente entre os registros daquele grupo. Esse nome é uma aproximação posterior, não um rótulo produzido originalmente pelo K-Means.

In [4]:
escalonador_cluster = StandardScaler()
dados_cluster = escalonador_cluster.fit_transform(df[FEATURES_CLUSTER])

modelo_cluster = KMeans(n_clusters=5, n_init=10, random_state=42)
grupos_cluster = modelo_cluster.fit_predict(dados_cluster)

silhueta = silhouette_score(dados_cluster, grupos_cluster)
ari = adjusted_rand_score(df[COLUNA_ALVO], grupos_cluster)
comparacao_cluster = pd.crosstab(
    pd.Series(grupos_cluster, name="grupo"),
    df[COLUNA_ALVO],
)

mapa_perfis_cluster = {}
for grupo in comparacao_cluster.index:
    mapa_perfis_cluster[int(grupo)] = str(
        comparacao_cluster.loc[grupo].idxmax()
    )

print("Silhueta:", round(silhueta, 3))
print("Adjusted Rand Index frente aos perfis de referência:", round(ari, 3))
print("\nDistribuição dos perfis de referência por grupo:")
print(comparacao_cluster.to_string())
print("\nNome aproximado de cada grupo:", mapa_perfis_cluster)

Silhueta: 0.404
Adjusted Rand Index frente aos perfis de referência: 0.889

Distribuição dos perfis de referência por grupo:
perfil_referencia  CONSUMO_CONSCIENTE  EXPLORADOR_SUSTENTAVEL  GUARDIAO_RECICLAGEM  GUARDIAO_RECURSOS  MOBILIZADOR_VERDE
grupo                                                                                                                   
0                                   0                      63                    1                  0                  0
1                                   0                       2                    0                  0                 80
2                                   0                       4                    0                 79                  0
3                                   0                       4                   79                  0                  0
4                                  80                       7                    0                  1                  0

Nome aproximado de cada gru

### Discussão do K-Means

Nesta base, a silhueta foi aproximadamente **0,404** e o ARI aproximadamente **0,889**. Os grupos se aproximam dos perfis de referência, mas não coincidem completamente: em especial, `EXPLORADOR_SUSTENTAVEL` aparece dividido entre grupos. Isso mostra que os cinco grupos não representam categorias perfeitas e que o nome atribuído a cada grupo deve ser tratado como uma descrição aproximada. Os números dos grupos podem mudar em outro treinamento.

## Modelo 2 — Random Forest para classificar perfil

A Random Forest classifica `perfil_referencia` usando as seis quantidades de missões por categoria. Uma divisão estratificada reserva 20% dos registros para teste e mantém a proporção de cada classe. O identificador do usuário e os totais derivados não entram como características. A acurácia, o relatório por classe e a matriz de confusão mostram o resultado geral e os perfis que podem ser confundidos.

In [7]:
X = df[FEATURES_PERFIL]
y = df[COLUNA_ALVO]
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

modelo_perfil = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
)
modelo_perfil.fit(X_treino, y_treino)
previsoes = modelo_perfil.predict(X_teste)
acuracia = accuracy_score(y_teste, previsoes)

print("Acurácia no conjunto de teste:", round(acuracia, 4))
print("\nRelatório por perfil:")
print(classification_report(y_teste, previsoes, zero_division=0))
matriz_confusao = pd.DataFrame(
    confusion_matrix(y_teste, previsoes, labels=modelo_perfil.classes_),
    index=modelo_perfil.classes_,
    columns=modelo_perfil.classes_,
)
print("Matriz de confusão (linhas: real; colunas: previsto):")
print(matriz_confusao.to_string())

Acurácia no conjunto de teste: 0.975

Relatório por perfil:
                        precision    recall  f1-score   support

    CONSUMO_CONSCIENTE       1.00      1.00      1.00        16
EXPLORADOR_SUSTENTAVEL       1.00      0.88      0.93        16
   GUARDIAO_RECICLAGEM       0.94      1.00      0.97        16
     GUARDIAO_RECURSOS       0.94      1.00      0.97        16
     MOBILIZADOR_VERDE       1.00      1.00      1.00        16

              accuracy                           0.97        80
             macro avg       0.98      0.97      0.97        80
          weighted avg       0.98      0.97      0.97        80

Matriz de confusão (linhas: real; colunas: previsto):
                        CONSUMO_CONSCIENTE  EXPLORADOR_SUSTENTAVEL  GUARDIAO_RECICLAGEM  GUARDIAO_RECURSOS  MOBILIZADOR_VERDE
CONSUMO_CONSCIENTE                      16                       0                    0                  0                  0
EXPLORADOR_SUSTENTAVEL                   0             

### Discussão da Random Forest

Nesta divisão, a Random Forest obteve **0,975 de acurácia**, classificando corretamente 78 dos 80 registros de teste. Os dois erros ocorreram no perfil `EXPLORADOR_SUSTENTAVEL`, confundido uma vez com `GUARDIAO_RECICLAGEM` e uma vez com `GUARDIAO_RECURSOS`. O resultado é alto para esta base, mas pode refletir a forma como os perfis de referência foram construídos. Deve ser reavaliado com dados reais antes de ser usado para orientar decisões sobre usuários.

## Modelo 3 — Regressão Linear para estimar pontuação

A Regressão Linear estima a pontuação do usuário a partir das seis quantidades de missões por categoria e da taxa de conclusão. Diferentemente da classificação, a saída é um valor numérico contínuo. A avaliação utiliza uma separação entre treino e teste e apresenta MAE, RMSE e R².

A variável-alvo `pontos` já existe no CSV. Como a origem da base não está documentada, a estimativa é experimental e não deve ser usada como pontuação oficial do usuário. No sistema real, a pontuação oficial continua sendo calculada e armazenada pelas regras do Java.


In [ ]:
FEATURES_REGRESSAO = FEATURES_CLUSTER
COLUNA_PONTOS = "pontos"

if COLUNA_PONTOS not in df.columns:
    raise ValueError("A base não possui a coluna pontos para treinar a regressão.")

X_regressao = df[FEATURES_REGRESSAO]
y_regressao = df[COLUNA_PONTOS]

X_reg_treino, X_reg_teste, y_reg_treino, y_reg_teste = train_test_split(
    X_regressao,
    y_regressao,
    test_size=0.2,
    random_state=42,
)

modelo_regressao = LinearRegression()
modelo_regressao.fit(X_reg_treino, y_reg_treino)
previsoes_pontos = modelo_regressao.predict(X_reg_teste)

mae_regressao = mean_absolute_error(y_reg_teste, previsoes_pontos)
rmse_regressao = mean_squared_error(y_reg_teste, previsoes_pontos) ** 0.5
r2_regressao = r2_score(y_reg_teste, previsoes_pontos)

print("MAE:", round(mae_regressao, 2))
print("RMSE:", round(rmse_regressao, 2))
print("R²:", round(r2_regressao, 4))


## Limitações e próximos passos

- O CSV tem 400 registros balanceados entre cinco perfis, mas não documenta a origem dos dados nem como os rótulos foram atribuídos.
- O K-Means é não supervisionado; os nomes dos grupos são uma interpretação feita depois do agrupamento usando os perfis de referência.
- A avaliação da Random Forest usa uma única divisão treino/teste. Novos dados ou divisões podem produzir outras métricas.
- Os resultados são exploratórios e não demonstram causalidade nem desempenho garantido para usuários reais.
- Uma próxima validação deve usar dados reais coletados no sistema, com consentimento e critérios de perfil documentados.

## Serialização dos modelos

O arquivo `modelos_soulup.joblib` contém K-Means, o escalonador ajustado, o nome aproximado de cada grupo, a Random Forest e os nomes e a ordem das características esperadas pela API. A API Flask carrega esse arquivo para gerar previsões sem executar o treinamento novamente.

In [11]:
pacote_modelos = {
    "modelo_cluster": modelo_cluster,
    "escalonador_cluster": escalonador_cluster,
    "mapa_perfis_cluster": mapa_perfis_cluster,
    "features_cluster": FEATURES_CLUSTER,
    "modelo_perfil": modelo_perfil,
    "modelo_regressao": modelo_regressao,
    "features_perfil": FEATURES_PERFIL,
    "features_regressao": FEATURES_REGRESSAO,
    "classes_perfil": [str(classe) for classe in modelo_perfil.classes_],
    "metrica_silhueta": float(silhueta),
    "metrica_ari": float(ari),
    "metrica_acuracia_teste": float(acuracia),
    "metrica_mae_regressao": float(mae_regressao),
    "metrica_rmse_regressao": float(rmse_regressao),
    "metrica_r2_regressao": float(r2_regressao),
}
joblib.dump(pacote_modelos, ARQUIVO_MODELOS)
print("Modelos salvos em:", ARQUIVO_MODELOS)

Modelos salvos em: /home/artur-oliveira/Documentos/challenge/ia/modelos_soulup.joblib
